# Libraries

In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pyxlsb import open_workbook
import warnings

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

SEED = 42
np.random.seed(SEED)

# Data

In [2]:
DATA_PATH = os.path.abspath(os.path.join(os.path.dirname("__file__"), "..", "data/javelin101.xlsb"))
if os.path.exists(DATA_PATH) is False:
    print("File not found: {}".format(DATA_PATH))
    
CACHE = os.path.abspath(os.path.join(os.path.dirname("__file__"), "..", "cache"))

In [3]:
def read_sheet(sheet, wide=False):
    """Read one sheet of the .xlsb. Row 0 = table title, row 1 = header."""
    f = f"{CACHE}/{sheet}.pkl"
    if os.path.exists(f):
        return pd.read_pickle(f)
    with open_workbook(DATA_PATH) as wb, wb.get_sheet(sheet) as sh:
        rows = [[c.v for c in r] for r in sh.rows()]
    hdr = rows[1]; n = len(hdr)
    body = [(r + [None] * n)[:n] for r in rows[2:] if r and r[0] is not None]
    df = pd.DataFrame(body, columns=hdr)
    if wide:   # genes x patients tables
        df = df.set_index(hdr[0]).apply(pd.to_numeric, errors="coerce")
    else:
        df = df.dropna(how="all")
    df.to_pickle(f)
    return df

def by_id(df, key):
    d = df.rename(columns={key: "id"}).drop_duplicates("id").set_index("id")
    return d.apply(pd.to_numeric, errors="coerce")

clin_raw = read_sheet("S11_Clinical_data")
cd8      = by_id(read_sheet("S12_CD8_IHC_data"), "ID")
imm      = by_id(read_sheet("S14_ImmuneNet_decon_data"), "row_id")
path     = by_id(read_sheet("S16_Pathway_scores"), "row_id")
wes      = by_id(read_sheet("S18_WES_summary_stats"), "ID")
math_    = by_id(read_sheet("S19_MATH_scores"), "ANONID")
fcgr_raw = read_sheet("S20_FCGR_mutation_data").drop_duplicates("ID").set_index("ID")
expr     = read_sheet("S13_Gene_expression_TPM", wide=True)      # log2 TPM, genes x patients
mut      = read_sheet("S21_Gene_mutation_status", wide=True)     # 0/1, genes x patients
print("clinical", clin_raw.shape, "| CD8", cd8.shape, "| immune", imm.shape, "| pathways", path.shape,
      "| WES", wes.shape, "| expr", expr.shape, "| mut", mut.shape)

clinical (886, 8) | CD8 (841, 6) | immune (727, 22) | pathways (726, 58) | WES (841, 3) | expr (22955, 726) | mut (18243, 738)


# 1. Clinical table

In [4]:
clin = clin_raw.rename(columns={"ID": "id"}).copy()
clin = clin[clin.TRT01P.isin(["Avelumab+Axitinib", "Sunitinib"])]
for c in ["PFS_P", "PFS_P_CNSR", "AGE"]:
    clin[c] = pd.to_numeric(clin[c], errors="coerce")
clin = clin.dropna(subset=["PFS_P", "PFS_P_CNSR"]).drop_duplicates("id").set_index("id")

clin["time"]  = clin.PFS_P                         # months
clin["event"] = (1 - clin.PFS_P_CNSR).astype(int)  # 1 = progression/death, 0 = censored
clin["T"]     = (clin.TRT01P == "Avelumab+Axitinib").astype(int)   # treatment indicator
print(clin.groupby("TRT01P").agg(n=("time", "size"), events=("event", "sum"), median_time_observed=("time", "median")))

                     n  events  median_time_observed
TRT01P                                              
Avelumab+Axitinib  441     188              7.096509
Sunitinib          444     234              6.488706


# 2. Feature matrix

Joins everything on patient ID: age, sex, PD-L1, TCGA cluster, CD8 IHC, immune deconvolution, pathway scores, TMB, MATH, FCGR genotype, RCC driver mutations and curated immune/angiogenesis genes

Restricts to patients with RNA-seq, drops features that are mostly missing or constant, and median-imputes the rest

In [5]:
RCC_MUT = ["VHL", "PBRM1", "BAP1", "SETD2", "KDM5C", "TP53", "PTEN", "MTOR", "ARID1A", "KMT2C", "KMT2D", "PIK3CA", "NF2", "TERT"]
EXPR_GENES = ["PDCD1", "CD274", "CTLA4", "LAG3", "TIGIT", "HAVCR2", "CD8A", "GZMB", "PRF1", "IFNG", "CXCL9", "CXCL10",
              "FOXP3", "CD68", "CD163", "VEGFA", "KDR", "FLT1", "EPAS1", "CA9", "PDGFRB", "IL6", "TGFB1", "MKI67"]

F = pd.DataFrame(index=clin.index)
F["age"]    = clin.AGE
F["male"]   = (clin.SEX == "M").astype(float)
F["pdl1_pos"] = clin.PDL1FL.map({"Y": 1.0, "N": 0.0})            # NaN if unknown
for k in ["m1", "m2", "m3", "m4", "Other"]:
    F[f"tcga_{k}"] = (clin.TCGA_cluster == k).astype(float)

F = F.join(cd8.add_prefix("ihc_").rename(columns=str.lower))
F = F.join(imm.add_prefix("imm_"))
F = F.join(path.add_prefix("path_"))
F = F.join(wes[["nonsyn_var_MB"]].rename(columns={"nonsyn_var_MB": "TMB"}))
F["log_TMB"] = np.log1p(F.TMB)
F = F.join(math_.rename(columns={"MATH": "MATH"}))
for g in ["FCGR2A(RS1801274)", "FCGR3A(RS396991)"]:
    d = pd.get_dummies(fcgr_raw[g].replace("NA", np.nan), prefix=g.split("(")[0]).astype(float)
    F = F.join(d)

mut_use = [g for g in RCC_MUT if g in mut.index and mut.loc[g].mean() >= 0.03]
F = F.join(mut.loc[mut_use].T.add_prefix("mut_"))
F = F.join(expr.loc[[g for g in EXPR_GENES if g in expr.index]].T.add_prefix("expr_"))

# analysis cohort = randomised patients with RNA-seq (the biomarker-evaluable population)
has_rna = clin.index.isin(expr.columns)
F = F[has_rna]; clin_c = clin[has_rna].copy()
F = F.loc[:, F.notna().mean() > 0.6]                 # drop features that are mostly missing
F = F.loc[:, F.nunique() > 1]
print("cohort:", len(F), "patients |", F.shape[1], "features | missing cells: %.1f%%" % (100 * F.isna().mean().mean()))

X = F.fillna(F.median())      # simple median imputation (documented limitation)
T = clin_c["T"].values
time_ = clin_c.time.values
event = clin_c.event.values
feat_names = list(X.columns)
print(clin_c.groupby("TRT01P").agg(n=("time", "size"), events=("event", "sum")))

cohort: 726 patients | 131 features | missing cells: 0.7%
                     n  events
TRT01P                        
Avelumab+Axitinib  354     157
Sunitinib          372     201


# 3. Average causal effect of the combination (randomised comparison)

In [6]:
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.plotting import add_at_risk_counts

fig, ax = plt.subplots(figsize=(6.5, 4.2))
kms = []
for lab, col in [("Sunitinib", "#888"), ("Avelumab+Axitinib", "#c0392b")]:
    m = clin_c.TRT01P == lab
    km = KaplanMeierFitter().fit(clin_c.time[m], clin_c.event[m], label=f"{lab} (n={m.sum()})")
    km.plot_survival_function(ax=ax, color=col, ci_show=True); kms.append(km)
ax.set_xlabel("Months"); ax.set_ylabel("Progression-free probability"); ax.set_title("PFS by arm (biomarker-evaluable cohort)")
add_at_risk_counts(*kms, ax=ax); plt.tight_layout(); plt.show()

d = clin_c[["time", "event", "T"]].assign(age=F.age, male=F.male)
cph = CoxPHFitter().fit(d, "time", "event")
print(cph.summary.loc[["T", "age", "male"], ["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].round(3))

ImportError: scipy._cyutility does not export expected C function slice_memviewslice

### Restricted mean survival time (RMST) — a censoring-robust effect size that does not need proportional hazards
RMST difference = average extra progression-free months gained within the first year

In [ ]:
TAU = 12.0

def km_stat(t, e, tau=TAU, kind="rmst"):
    """Kaplan-Meier RMST up to tau, or S(tau)."""
    o = np.argsort(t, kind="mergesort"); t = t[o]; e = e[o]
    te = t[e == 1]
    ut = np.unique(te[te <= tau])
    if len(ut) == 0:
        return tau if kind == "rmst" else 1.0
    n_at = len(t) - np.searchsorted(t, ut, "left")
    d_ev = np.searchsorted(te, ut, "right") - np.searchsorted(te, ut, "left")
    S = np.cumprod(1 - d_ev / n_at)
    if kind == "surv":
        return S[-1]
    grid = np.r_[0, ut, tau]
    return float(np.sum(np.r_[1, S] * np.diff(grid)))

def boot_diff(t, e, tr, kind="rmst", B=500, seed=0):
    rng = np.random.default_rng(seed); n = len(t)
    est = km_stat(t[tr == 1], e[tr == 1], kind=kind) - km_stat(t[tr == 0], e[tr == 0], kind=kind)
    bs = []
    for _ in range(B):
        i = rng.integers(0, n, n)
        if tr[i].min() == tr[i].max(): continue
        bs.append(km_stat(t[i][tr[i] == 1], e[i][tr[i] == 1], kind=kind) - km_stat(t[i][tr[i] == 0], e[i][tr[i] == 0], kind=kind))
    return est, np.percentile(bs, [2.5, 97.5])

for kind, lab in [("rmst", "RMST(12m) difference (months)"), ("surv", "12-month PFS probability difference")]:
    est, ci = boot_diff(time_, event, T, kind)
    print(f"{lab}: {est:+.2f}  (95% CI {ci[0]:+.2f} to {ci[1]:+.2f})")

RMST(12m) difference (months): +1.20  (95% CI +0.59 to +1.76)
12-month PFS probability difference: +0.15  (95% CI +0.07 to +0.24)


# 4. Pseudo-observations: turning censored survival data into a regression-ready outcome

For each patient we compute a jackknife pseudo-observation of RMST(12m) and of S(12m) **within their own arm**. These behave like uncensored outcomes (their conditional mean equals the true quantity), so standard causal-ML estimators can be used without discarding censored patients.

In [ ]:
def pseudo_obs(t, e, tr, kind):
    po = np.zeros(len(t))
    for a in (0, 1):
        idx = np.where(tr == a)[0]; n = len(idx)
        full = km_stat(t[idx], e[idx], kind=kind)
        for j, i in enumerate(idx):
            keep = np.delete(idx, j)
            po[i] = n * full - (n - 1) * km_stat(t[keep], e[keep], kind=kind)
    return po

Y_rmst = pseudo_obs(time_, event, T, "rmst")     # months of PFS within first 12 months
Y_s12  = pseudo_obs(time_, event, T, "surv")     # probability of being progression-free at 12 months
print("mean pseudo-RMST  Sunitinib %.2f | Avelumab+Axitinib %.2f" % (Y_rmst[T == 0].mean(), Y_rmst[T == 1].mean()))
print("mean pseudo-S(12) Sunitinib %.3f | Avelumab+Axitinib %.3f" % (Y_s12[T == 0].mean(), Y_s12[T == 1].mean()))

mean pseudo-RMST  Sunitinib 7.79 | Avelumab+Axitinib 8.99
mean pseudo-S(12) Sunitinib 0.354 | Avelumab+Axitinib 0.509


# 5. Predicting survival (PFS)
Four models, 5-fold cross-validated Harrell C-index (higher = better; 0.5 = coin flip).
* Clinical only (arm, age, sex, TCGA cluster)
* Elastic-net Cox, all features + arm
* Elastic-net Cox with arm × feature interactions (lets the model give *different* predictions per arm)
* Random Survival Forest, all features + arm

In [ ]:
from sklearn.model_selection import KFold, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis, CoxPHSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored

y_surv = Surv.from_arrays(event.astype(bool), time_)

clin_cols = ["age", "male"] + [c for c in feat_names if c.startswith("tcga_")]
XA = np.c_[T, X[clin_cols].values]
XB = np.c_[T, X.values]
XC = np.c_[T, X.values, X.values * T[:, None]]            # arm x feature interactions (features standardised later)
XD = np.c_[T, X.values]
names_C = ["T"] + feat_names + [f"T:{c}" for c in feat_names]

def make_coxnet():
    base = make_pipeline(StandardScaler(), CoxnetSurvivalAnalysis(l1_ratio=0.5, alpha_min_ratio=0.05, max_iter=2000))
    return base

def fit_coxnet(Xtr, ytr, seed=0):
    est = make_coxnet().fit(Xtr, ytr)
    alphas = est[-1].alphas_
    gcv = GridSearchCV(
        make_pipeline(StandardScaler(), CoxnetSurvivalAnalysis(l1_ratio=0.5, fit_baseline_model=True, max_iter=2000)),
        {"coxnetsurvivalanalysis__alphas": [[a] for a in alphas]},
        cv=KFold(3, shuffle=True, random_state=seed), error_score=0.5, n_jobs=-1)
    return gcv.fit(Xtr, ytr)

strata = event * 2 + T
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
cidx = {k: [] for k in "ABCD"}
oof_risk = {k: np.zeros(len(T)) for k in "ABCD"}
for tr_i, te_i in skf.split(XA, strata):
    for k, Xm in zip("ABCD", [XA, XB, XC, XD]):
        if k == "A":
            mdl = make_pipeline(StandardScaler(), CoxPHSurvivalAnalysis(alpha=1.0)).fit(Xm[tr_i], y_surv[tr_i])
        elif k in "BC":
            mdl = fit_coxnet(Xm[tr_i], y_surv[tr_i])
        else:
            mdl = RandomSurvivalForest(n_estimators=300, min_samples_leaf=15, max_features="sqrt", n_jobs=-1, random_state=SEED).fit(Xm[tr_i], y_surv[tr_i])
        r = mdl.predict(Xm[te_i])
        oof_risk[k][te_i] = r
        cidx[k].append(concordance_index_censored(event[te_i].astype(bool), time_[te_i], r)[0])

labels = {"A": "A  Clinical only", "B": "B  Elastic-net Cox (main effects)", "C": "C  Elastic-net Cox (+ arm interactions)", "D": "D  Random Survival Forest"}
res = pd.DataFrame({labels[k]: [np.mean(v), np.std(v)] for k, v in cidx.items()}, index=["C-index mean", "sd"]).T
print(res.round(3))

ImportError: cannot import name 'HalfPoissonLossArrayAPI' from 'sklearn._loss.loss' (c:\Users\qt49\.conda\envs\xiao\Lib\site-packages\sklearn\_loss\loss.py)